# 01.1 GPU Memory Architecture & VRAM Engineering

**Topics:** VRAM calculation formulas, model size estimation from architecture params,
memory bandwidth analysis, OOM prediction, roofline bottleneck identification.

Uses real `torch.cuda` measurements when GPU available, graceful CPU fallback with catalog specs.

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Dict, List, Tuple

try:
    import torch
    HAS_CUDA = torch.cuda.is_available()
except ImportError:
    HAS_CUDA = False

from utils.gpu_info import GPU_CATALOG, detect_gpu, GPUInfo
from utils.roofline import plot_roofline

print(f"CUDA available: {HAS_CUDA}")
if HAS_CUDA:
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Model Size Estimation from Architecture Parameters

Calculate exact parameter count from transformer architecture dimensions.
Formula: `params = embedding + layers*(attention + FFN + norms) + head`

In [ ]:
@dataclass
class LlamaConfig:
    name: str
    hidden_dim: int
    num_layers: int
    num_heads: int
    num_kv_heads: int
    intermediate_dim: int
    vocab_size: int = 128256

MODELS = {
    '8B': LlamaConfig('Llama-3.1-8B', 4096, 32, 32, 8, 14336),
    '70B': LlamaConfig('Llama-3.1-70B', 8192, 80, 64, 8, 28672),
    '405B': LlamaConfig('Llama-3.1-405B', 16384, 126, 128, 8, 53248),
}

def estimate_params(cfg: LlamaConfig) -> Dict[str, int]:
    """Estimate parameter count from architecture dimensions."""
    d = cfg.hidden_dim
    head_dim = d // cfg.num_heads
    # Embedding + unembedding
    embed = cfg.vocab_size * d
    unembed = cfg.vocab_size * d
    # Per-layer attention: Q, K, V projections + output
    q_proj = d * (cfg.num_heads * head_dim)
    k_proj = d * (cfg.num_kv_heads * head_dim)
    v_proj = d * (cfg.num_kv_heads * head_dim)
    o_proj = (cfg.num_heads * head_dim) * d
    attn_per_layer = q_proj + k_proj + v_proj + o_proj
    # Per-layer FFN (SwiGLU): gate + up + down
    ffn_per_layer = 3 * d * cfg.intermediate_dim
    # Per-layer norms (RMSNorm): 2 * hidden_dim
    norm_per_layer = 2 * d
    # Total
    total_layers = cfg.num_layers * (attn_per_layer + ffn_per_layer + norm_per_layer)
    final_norm = d
    total = embed + total_layers + final_norm + unembed
    return {'embedding': embed, 'unembed': unembed,
            'attn_per_layer': attn_per_layer, 'ffn_per_layer': ffn_per_layer,
            'total_layers': total_layers, 'total': total,
            'total_billions': total / 1e9}

print(f"{'Model':<8} {'Estimated':>12} {'Attn/Layer':>12} {'FFN/Layer':>12} {'Embed':>10}")
print('-' * 58)
for name, cfg in MODELS.items():
    p = estimate_params(cfg)
    print(f"{name:<8} {p['total_billions']:>10.2f}B {p['attn_per_layer']/1e6:>10.1f}M "
          f"{p['ffn_per_layer']/1e6:>10.1f}M {p['embedding']/1e6:>8.1f}M")

## 2. VRAM Calculation Formulas

Four components of inference VRAM:
1. **Weights** = params × bytes_per_param
2. **KV Cache** = 2 × layers × kv_heads × head_dim × seq_len × batch × kv_bytes
3. **Activations** = batch × seq × hidden × intermediate_factor × bytes
4. **Overhead** = CUDA context (~300-800MB) + fragmentation (~10%)

In [ ]:
BYTES_PER_PARAM = {'fp32': 4, 'fp16': 2, 'bf16': 2, 'int8': 1, 'int4': 0.5, 'fp8': 1}

def calc_vram_gb(cfg: LlamaConfig, precision: str = 'fp16',
                 seq_len: int = 2048, batch_size: int = 1,
                 kv_precision: str = None) -> Dict[str, float]:
    """Calculate VRAM breakdown in GB."""
    kv_precision = kv_precision or precision
    bpp = BYTES_PER_PARAM[precision]
    kv_bpp = BYTES_PER_PARAM[kv_precision]
    params = estimate_params(cfg)['total']
    head_dim = cfg.hidden_dim // cfg.num_heads

    weights_gb = params * bpp / (1024**3)
    kv_cache_gb = (2 * cfg.num_layers * cfg.num_kv_heads * head_dim
                   * seq_len * batch_size * kv_bpp) / (1024**3)
    # Peak activations: attention scores + FFN intermediate
    attn_act = batch_size * cfg.num_heads * seq_len * seq_len * 2  # fp16 scores
    ffn_act = batch_size * seq_len * cfg.intermediate_dim * 2  # fp16
    activations_gb = (attn_act + ffn_act) / (1024**3)
    overhead_gb = 0.5 + (weights_gb + kv_cache_gb) * 0.10  # CUDA ctx + 10% frag

    total = weights_gb + kv_cache_gb + activations_gb + overhead_gb
    return {'weights': weights_gb, 'kv_cache': kv_cache_gb,
            'activations': activations_gb, 'overhead': overhead_gb, 'total': total}

# Full breakdown table
print(f"{'Model':<6} {'Prec':<5} {'Weights':>8} {'KV Cache':>9} {'Act':>6} {'Overhead':>8} {'Total':>7} GB")
print('-' * 62)
for name, cfg in MODELS.items():
    for prec in ['fp16', 'int8', 'int4']:
        v = calc_vram_gb(cfg, prec, seq_len=4096, batch_size=1)
        print(f"{name:<6} {prec:<5} {v['weights']:>8.1f} {v['kv_cache']:>9.2f} "
              f"{v['activations']:>6.2f} {v['overhead']:>8.2f} {v['total']:>7.1f}")

## 3. Real GPU Memory Measurement

Measure actual VRAM consumption by allocating tensors of known size.
Validates our formulas against hardware reality.

In [ ]:
def measure_actual_vram():
    """Measure real VRAM usage by allocating model-sized tensors."""
    if not HAS_CUDA:
        print("No GPU -- using catalog specs for A100-80GB as reference.")
        return {'total_gb': 80.0, 'free_gb': 79.0, 'used_gb': 1.0,
                'cuda_overhead_mb': 500, 'measured': False}

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    baseline = torch.cuda.memory_allocated()

    # Allocate 1GB tensor to measure overhead
    x = torch.zeros(256, 1024, 1024, dtype=torch.float16, device='cuda')  # 512MB
    after_alloc = torch.cuda.memory_allocated()
    actual_512mb = (after_alloc - baseline) / 1e6
    del x
    torch.cuda.empty_cache()

    props = torch.cuda.get_device_properties(0)
    total = props.total_mem / 1e9
    free = (props.total_mem - torch.cuda.memory_allocated()) / 1e9
    cuda_overhead = torch.cuda.memory_allocated() / 1e6

    print(f"Total VRAM: {total:.1f} GB")
    print(f"Free VRAM: {free:.1f} GB")
    print(f"CUDA context overhead: {cuda_overhead:.0f} MB")
    print(f"512MB allocation actual: {actual_512mb:.1f} MB (expect ~512)")
    return {'total_gb': total, 'free_gb': free, 'used_gb': total - free,
            'cuda_overhead_mb': cuda_overhead, 'measured': True}

gpu_mem = measure_actual_vram()

## 4. Memory Bandwidth Analysis

Measure effective memory bandwidth via large tensor copies.
Compare to theoretical peak to understand real-world efficiency.

In [ ]:
def measure_bandwidth(size_gb: float = 1.0, n_iters: int = 20) -> Dict[str, float]:
    """Measure effective HBM bandwidth via tensor copy."""
    if not HAS_CUDA:
        # Return theoretical specs for analysis
        print("CPU fallback: using H100 theoretical bandwidth (3350 GB/s)")
        return {'effective_gbs': 3015, 'theoretical_gbs': 3350, 'efficiency': 0.90}

    n_elements = int(size_gb * 1024**3 / 2)  # fp16 = 2 bytes
    src = torch.randn(n_elements, dtype=torch.float16, device='cuda')
    dst = torch.empty_like(src)

    # Warmup
    for _ in range(5):
        dst.copy_(src)
    torch.cuda.synchronize()

    # Timed runs
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(n_iters):
        dst.copy_(src)
    end.record()
    torch.cuda.synchronize()

    elapsed_ms = start.elapsed_time(end)
    bytes_moved = 2 * n_elements * 2 * n_iters  # read + write
    effective_gbs = bytes_moved / (elapsed_ms / 1000) / 1e9

    # Get theoretical from catalog
    gpu_name = torch.cuda.get_device_name(0)
    theoretical = 2039  # default A100
    for key, specs in GPU_CATALOG.items():
        if key.lower() in gpu_name.lower():
            theoretical = specs['bw_gbs']
            break

    del src, dst
    torch.cuda.empty_cache()

    print(f"Effective bandwidth: {effective_gbs:.0f} GB/s")
    print(f"Theoretical peak: {theoretical} GB/s")
    print(f"Efficiency: {effective_gbs/theoretical*100:.1f}%")
    return {'effective_gbs': effective_gbs, 'theoretical_gbs': theoretical,
            'efficiency': effective_gbs / theoretical}

bw_result = measure_bandwidth()

## 5. KV Cache Scaling: Batch Size × Sequence Length

Visualize how KV cache dominates VRAM at scale. This is the key insight:
weights are fixed cost, KV cache grows linearly with batch×seq.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: batch size scaling (fixed seq=4096)
cfg = MODELS['70B']
batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128]
components = {k: [] for k in ['weights', 'kv_cache', 'activations', 'overhead']}
for bs in batch_sizes:
    v = calc_vram_gb(cfg, 'fp16', seq_len=4096, batch_size=bs)
    for k in components:
        components[k].append(v[k])

ax = axes[0]
bottom = np.zeros(len(batch_sizes))
colors = ['#2563eb', '#dc2626', '#16a34a', '#9ca3af']
for (k, vals), c in zip(components.items(), colors):
    ax.bar(range(len(batch_sizes)), vals, bottom=bottom, label=k, color=c, alpha=0.85)
    bottom += np.array(vals)
ax.set_xticks(range(len(batch_sizes)))
ax.set_xticklabels(batch_sizes)
ax.set_xlabel('Batch Size'); ax.set_ylabel('VRAM (GB)')
ax.set_title('Llama-70B fp16 VRAM vs Batch Size (seq=4096)')
ax.legend(loc='upper left'); ax.axhline(80, color='red', linestyle='--', alpha=0.5, label='80GB limit')

# Right: sequence length scaling (fixed batch=8)
seq_lens = [512, 1024, 2048, 4096, 8192, 16384, 32768]
kv_gb = [calc_vram_gb(cfg, 'fp16', seq_len=s, batch_size=8)['kv_cache'] for s in seq_lens]
total_gb = [calc_vram_gb(cfg, 'fp16', seq_len=s, batch_size=8)['total'] for s in seq_lens]

ax = axes[1]
ax.semilogy(seq_lens, kv_gb, 'ro-', label='KV Cache', linewidth=2)
ax.semilogy(seq_lens, total_gb, 'b^-', label='Total VRAM', linewidth=2)
ax.axhline(80, color='gray', linestyle='--', alpha=0.5, label='80GB (1×H100)')
ax.axhline(640, color='gray', linestyle=':', alpha=0.5, label='640GB (8×H100)')
ax.set_xlabel('Sequence Length'); ax.set_ylabel('VRAM (GB)')
ax.set_title('Llama-70B fp16 VRAM vs Seq Length (batch=8)')
ax.legend()

plt.tight_layout(); plt.show()

## 6. OOM Prediction Engine

Given a GPU VRAM budget, predict the maximum batch size and sequence length
before hitting OOM. Critical for capacity planning.

In [ ]:
def predict_oom(cfg: LlamaConfig, precision: str, vram_budget_gb: float,
                kv_precision: str = None) -> Dict[str, any]:
    """Find max batch_size × seq_len before OOM."""
    results = []
    for seq_len in [512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072]:
        max_bs = 0
        for bs in range(1, 513):
            v = calc_vram_gb(cfg, precision, seq_len, bs, kv_precision)
            if v['total'] > vram_budget_gb:
                break
            max_bs = bs
        if max_bs > 0:
            results.append({'seq_len': seq_len, 'max_batch': max_bs,
                            'total_gb': calc_vram_gb(cfg, precision, seq_len, max_bs, kv_precision)['total'],
                            'throughput_proxy': max_bs * seq_len})
    return results

# OOM boundaries for Llama-8B on different GPUs
print("=== OOM Boundaries: Llama-8B fp16 ===")
print(f"{'GPU VRAM':<10} {'Seq Len':>8} {'Max Batch':>10} {'Used GB':>8} {'Throughput':>11}")
print('-' * 52)
for vram, gpu_name in [(24, 'A10G'), (48, 'L40S'), (80, 'H100')]:
    oom = predict_oom(MODELS['8B'], 'fp16', vram)
    for r in oom:
        print(f"{gpu_name+f' ({vram}GB)':<10} {r['seq_len']:>8} {r['max_batch']:>10} "
              f"{r['total_gb']:>8.1f} {r['throughput_proxy']:>11,}")
    print()

In [ ]:
# Visualize OOM boundaries as heatmap
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (model_name, cfg) in zip(axes, [('8B', MODELS['8B']), ('70B', MODELS['70B']), ('405B', MODELS['405B'])]):
    seq_lens = [512, 1024, 2048, 4096, 8192, 16384]
    batch_sizes = [1, 2, 4, 8, 16, 32, 64]
    grid = np.zeros((len(batch_sizes), len(seq_lens)))
    for i, bs in enumerate(batch_sizes):
        for j, sl in enumerate(seq_lens):
            grid[i, j] = calc_vram_gb(cfg, 'fp16', sl, bs)['total']

    im = ax.imshow(grid, aspect='auto', cmap='RdYlGn_r', vmin=0, vmax=640)
    ax.set_xticks(range(len(seq_lens))); ax.set_xticklabels(seq_lens, fontsize=8)
    ax.set_yticks(range(len(batch_sizes))); ax.set_yticklabels(batch_sizes)
    ax.set_xlabel('Seq Length'); ax.set_ylabel('Batch Size')
    ax.set_title(f'Llama-{model_name} fp16 VRAM (GB)')
    # OOM contour lines
    for threshold, color in [(24, 'blue'), (80, 'orange'), (640, 'red')]:
        ax.contour(grid, levels=[threshold], colors=[color], linewidths=2)
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('VRAM Heatmaps (contours: 24GB=blue, 80GB=orange, 640GB=red)', fontsize=11)
plt.tight_layout(); plt.show()

## 7. Roofline Bottleneck Analysis

Determine whether decode is memory-bound or compute-bound.
Uses arithmetic intensity: AI = 2×batch (FLOPs/byte for decode).

In [ ]:
GPU_SPECS = {
    'A10G': {'tflops': 125, 'bw_gbs': 600},
    'A100': {'tflops': 312, 'bw_gbs': 2039},
    'H100': {'tflops': 990, 'bw_gbs': 3350},
}

def decode_bottleneck(cfg: LlamaConfig, precision: str, batch_size: int,
                      gpu: str = 'H100') -> Dict[str, float]:
    """Roofline analysis for decode phase."""
    bpp = BYTES_PER_PARAM[precision]
    params = estimate_params(cfg)['total']
    specs = GPU_SPECS[gpu]
    # Decode: read all weights once per token, do 2*params*batch FLOPs
    bytes_per_step = params * bpp
    flops_per_step = 2 * params * batch_size
    ai = flops_per_step / bytes_per_step  # = 2*batch/bpp for fp16 -> batch
    ridge = specs['tflops'] * 1e3 / specs['bw_gbs']  # FLOP/byte
    bound = 'compute' if ai >= ridge else 'memory'
    # Achievable throughput
    if bound == 'memory':
        achievable_flops = ai * specs['bw_gbs'] * 1e9
    else:
        achievable_flops = specs['tflops'] * 1e12
    tokens_per_sec = achievable_flops / (2 * params)
    utilization = min(ai / ridge, 1.0)
    return {'ai': ai, 'ridge': ridge, 'bound': bound,
            'utilization': utilization, 'tokens_per_sec': tokens_per_sec}

# Find crossover batch size for each model
print(f"{'Model':<6} {'GPU':<6} {'BS':>4} {'AI':>8} {'Ridge':>7} {'Bound':<8} {'Util%':>6} {'tok/s':>8}")
print('-' * 60)
for model_name in ['8B', '70B']:
    for bs in [1, 4, 16, 64, 128, 256, 512]:
        r = decode_bottleneck(MODELS[model_name], 'fp16', bs, 'H100')
        print(f"{model_name:<6} {'H100':<6} {bs:>4} {r['ai']:>8.1f} {r['ridge']:>7.1f} "
              f"{r['bound']:<8} {r['utilization']*100:>5.1f}% {r['tokens_per_sec']:>8.0f}")

In [ ]:
# Roofline plot with workload points
fig, ax = plt.subplots(figsize=(10, 6))
gpu_name = 'H100'
specs = GPU_SPECS[gpu_name]
peak_gflops = specs['tflops'] * 1000
bw = specs['bw_gbs']
ridge = peak_gflops / bw

ai_range = np.logspace(-1, 4, 300)
roofline = np.minimum(peak_gflops, ai_range * bw)
ax.loglog(ai_range, roofline, 'k-', linewidth=2.5, label=f'{gpu_name} Roofline')
ax.axvline(ridge, color='gray', linestyle='--', alpha=0.6, label=f'Ridge={ridge:.0f} FLOP/B')
ax.fill_between(ai_range, 0.1, roofline, where=(ai_range < ridge), alpha=0.06, color='red')
ax.fill_between(ai_range, 0.1, roofline, where=(ai_range >= ridge), alpha=0.06, color='green')

# Plot workload points
markers = {'8B': 'o', '70B': 's'}
colors_map = {1: '#ef4444', 16: '#f59e0b', 64: '#22c55e', 256: '#3b82f6', 512: '#8b5cf6'}
for model_name in ['8B', '70B']:
    for bs, color in colors_map.items():
        r = decode_bottleneck(MODELS[model_name], 'fp16', bs, gpu_name)
        achieved = min(peak_gflops, r['ai'] * bw)
        ax.plot(r['ai'], achieved, markers[model_name], color=color, markersize=9)
        ax.annotate(f"{model_name} bs={bs}", (r['ai'], achieved),
                    fontsize=7, textcoords='offset points', xytext=(5, 5))

ax.set_xlabel('Arithmetic Intensity (FLOP/Byte)', fontsize=11)
ax.set_ylabel('Performance (GFLOP/s)', fontsize=11)
ax.set_title(f'{gpu_name} Roofline: LLM Decode Phase', fontsize=13)
ax.legend(loc='lower right'); ax.grid(True, alpha=0.3)
ax.set_xlim(0.5, 5000); ax.set_ylim(100, peak_gflops * 2)
plt.tight_layout(); plt.show()

## 8. Bandwidth-Limited Token Generation Rate

For memory-bound decode (batch=1), throughput is entirely determined by
how fast you can stream weights through HBM: `tokens/s = bandwidth / model_bytes`

In [ ]:
def bandwidth_limited_throughput(cfg: LlamaConfig, precision: str, gpu: str) -> float:
    """Max tokens/s at batch=1 (pure memory-bound)."""
    params = estimate_params(cfg)['total']
    model_bytes = params * BYTES_PER_PARAM[precision]
    bw_bytes_per_sec = GPU_SPECS[gpu]['bw_gbs'] * 1e9
    return bw_bytes_per_sec / model_bytes

print("=== Bandwidth-Limited Decode (batch=1) ===")
print(f"{'Model':<6} {'Precision':<6} {'GPU':<6} {'Model Size':>10} {'tok/s':>8}")
print('-' * 42)
for model_name, cfg in MODELS.items():
    for prec in ['fp16', 'int8', 'int4']:
        for gpu in ['A100', 'H100']:
            tps = bandwidth_limited_throughput(cfg, prec, gpu)
            size_gb = estimate_params(cfg)['total'] * BYTES_PER_PARAM[prec] / 1e9
            print(f"{model_name:<6} {prec:<6} {gpu:<6} {size_gb:>8.1f}GB {tps:>8.1f}")

## 9. Quantization Impact: Memory vs Quality Tradeoff

Compare VRAM savings from quantization against the throughput gains.
INT4 gives 4× memory reduction but also 4× higher bandwidth-limited throughput.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

precisions = ['fp16', 'int8', 'int4']
models_to_plot = ['8B', '70B', '405B']
x = np.arange(len(models_to_plot))
width = 0.25

# Left: VRAM comparison
for i, prec in enumerate(precisions):
    vrams = [calc_vram_gb(MODELS[m], prec, 4096, 1)['total'] for m in models_to_plot]
    ax1.bar(x + i*width, vrams, width, label=prec, alpha=0.85)
ax1.set_xticks(x + width); ax1.set_xticklabels([f'Llama-{m}' for m in models_to_plot])
ax1.set_ylabel('VRAM (GB)'); ax1.set_title('VRAM by Precision (batch=1, seq=4096)')
ax1.axhline(80, color='red', linestyle='--', alpha=0.5, label='80GB limit')
ax1.legend(); ax1.set_yscale('log')

# Right: Throughput comparison
for i, prec in enumerate(precisions):
    tps = [bandwidth_limited_throughput(MODELS[m], prec, 'H100') for m in models_to_plot]
    ax2.bar(x + i*width, tps, width, label=prec, alpha=0.85)
ax2.set_xticks(x + width); ax2.set_xticklabels([f'Llama-{m}' for m in models_to_plot])
ax2.set_ylabel('Tokens/sec (batch=1)'); ax2.set_title('BW-Limited Throughput on H100')
ax2.legend()

plt.tight_layout(); plt.show()

## Key Takeaways

1. **Weights are fixed cost** -- Llama-70B fp16 = ~131GB regardless of batch/seq
2. **KV cache scales linearly** with batch×seq -- dominates at high concurrency
3. **Decode is memory-bound** at small batch (AI = batch for fp16) -- crossover ~295 on H100
4. **Bandwidth determines single-user latency** -- INT4 gives 4× speedup at batch=1
5. **OOM prediction** enables capacity planning: know your limits before deployment
6. **Quantization is free throughput** -- INT4 fits 70B on single 24GB GPU with 4× speed